# SmolVLM-256M: Image Preprocessing on the GPU

This notebook compares three image processors: the slow one, the fast one on the CPU, and the fast one on the GPU. It checks the time, the pixel values and the answer.

In [ ]:
import copy
import gc
import json
import re
import subprocess
import time
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor, StaticCache
from transformers.modeling_outputs import BaseModelOutput

ROOT = next(p for p in (Path(".."), Path("."), Path("/content/edge-ai-smol-lvm-250-optimization"))
            if (p / "frame.jpg").exists())
IMAGE = ROOT / "frame.jpg"
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
PROMPT = "Describe any visible defects."
PREPROCESS = {"default": (None, "cpu"), "slow": (False, "cpu"), "fast-cpu": (True, "cpu"), "fast-gpu": (True, "cuda")}


def timed(fn):
    torch.cuda.synchronize()
    t = time.perf_counter()
    out = fn()
    torch.cuda.synchronize()
    return out, (time.perf_counter() - t) * 1e3


def mean_ms(fn, runs=5):
    fn()
    torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(runs):
        fn()
        torch.cuda.synchronize()
    return (time.perf_counter() - t) * 1e3 / runs


def until_eos(ids, eos):
    for i, t in enumerate(ids):
        if t in eos:
            return ids[:i]
    return ids


def set_attn(model, impl):
    if hasattr(model, "set_attn_implementation"):
        model.set_attn_implementation(impl)
    else:
        for c in (model.config.text_config, model.config.vision_config):
            c._attn_implementation = impl


def load(attn=None, fast=None):
    kw = {} if attn is None else {"attn_implementation": attn}
    proc = AutoProcessor.from_pretrained(MODEL_ID, **({} if fast is None else {"use_fast": fast}))
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float16, **kw).to("cuda").eval()
    return model, proc


def eos_ids(tok):
    return {tok.eos_token_id, tok.convert_tokens_to_ids("<end_of_utterance>")}


def chat_text(proc, prompt):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": prompt}]}]
    return proc.apply_chat_template(msgs, add_generation_prompt=True)


def make_inputs(proc, image, prompt, image_device="cpu", size=None, extra=None):
    img = Image.open(image).convert("RGB") if isinstance(image, (str, Path)) else image
    dev = {} if image_device == "cpu" else {"device": image_device}
    ik = {**({} if size is None else {"size": {"longest_edge": size}}), **dev, **(extra or {})}
    text = chat_text(proc, prompt)
    try:
        out = proc(text=text, images=[img], return_tensors="pt", images_kwargs=ik)
    except (TypeError, ValueError):
        out = proc(text=text, images=[img], return_tensors="pt", **ik)
    out = out.to("cuda")
    out["pixel_values"] = out["pixel_values"].to(torch.float16)
    return out


def bare_generate(model, inputs, k, stop=True):
    extra = {} if stop else {"min_new_tokens": k}
    with torch.inference_mode():
        return model.generate(**inputs, max_new_tokens=k, do_sample=False, **extra)


class GraphDecoder:
    def __init__(self, model, max_new=64, max_len=4096, own_mask=False):
        self.model, self.max_new, self.max_len = model, max_new, max_len
        self.cache = StaticCache(config=model.config.text_config, max_cache_len=max_len)
        dev = model.device
        self.ids = torch.zeros(1, 1, dtype=torch.long, device=dev)
        self.pos = torch.zeros(1, dtype=torch.long, device=dev)
        self.idx = torch.zeros(1, dtype=torch.long, device=dev)
        self.out = torch.zeros(max_new, dtype=torch.long, device=dev)
        self.graph = None
        self.steps = max_new
        self.own_mask = own_mask or model.config.text_config._attn_implementation == "eager"

    def mask(self):
        allowed = torch.arange(self.max_len, device=self.pos.device) <= self.pos[:, None]
        dt = self.model.dtype
        return ((~allowed).to(dt) * torch.finfo(dt).min)[:, None, None, :]

    def step(self):
        extra = {"attention_mask": self.mask()} if self.own_mask else {}
        h = self.model.model.text_model(
            input_ids=self.ids, cache_position=self.pos, past_key_values=self.cache, use_cache=True, **extra
        ).last_hidden_state
        nxt = self.model.lm_head(h[:, -1]).argmax(-1, keepdim=True)
        self.ids.copy_(nxt)
        self.out.index_copy_(0, self.idx, nxt.view(1))
        self.pos.add_(1)
        self.idx.add_(1)

    @torch.inference_mode()
    def capture(self):
        s = torch.cuda.Stream()
        s.wait_stream(torch.cuda.current_stream())
        with torch.cuda.stream(s):
            for _ in range(3):
                self.pos.fill_(self.max_len - 1)
                self.idx.zero_()
                self.step()
        torch.cuda.current_stream().wait_stream(s)
        self.pos.fill_(self.max_len - 1)
        self.idx.zero_()
        self.graph = torch.cuda.CUDAGraph()
        with torch.cuda.graph(self.graph):
            self.step()

    @torch.inference_mode()
    def prefill(self, inputs):
        self.cache.reset()
        n = inputs["input_ids"].shape[1]
        logits = self.model(
            **inputs, past_key_values=self.cache, use_cache=True,
            cache_position=torch.arange(n, device=self.model.device),
        ).logits
        first = logits[:, -1].argmax(-1, keepdim=True)
        self.ids.copy_(first)
        self.pos.fill_(n)
        self.idx.zero_()
        return first

    @torch.inference_mode()
    def decode(self, use_graph=True, eos=None):
        self.steps = self.max_new
        for i in range(self.max_new):
            if use_graph:
                self.graph.replay()
            else:
                self.step()
            if eos and self.ids.item() in eos:
                self.steps = i + 1
                break
        return self.out.clone()

    def generate(self, inputs, eos, use_graph=True):
        first = self.prefill(inputs)
        ids = torch.cat([first.view(1), self.decode(use_graph, eos)]).tolist()
        return until_eos(ids, eos)


def time_decode(dec, inputs, eos, runs=5, use_graph=True, stop=False):
    pre, dec_ms, ids = [], [], None
    for i in range(runs + 1):
        first, p = timed(lambda: dec.prefill(inputs))
        out, d = timed(lambda: dec.decode(use_graph, eos if stop else None))
        if i:
            pre.append(p)
            dec_ms.append(d)
        ids = until_eos(torch.cat([first.view(1), out]).tolist(), eos)
    return {"prefill_ms": float(np.mean(pre)), "decode_ms": float(np.mean(dec_ms)), "steps": dec.steps, "ids": ids}


def fit_len(n, max_new):
    return -(-(n + max_new + 1) // 64) * 64

In [ ]:
model, _ = load()
eos = eos_ids(AutoProcessor.from_pretrained(MODEL_ID).tokenizer)
dec = GraphDecoder(model, 64, 1024)
dec.capture()
img = Image.open(IMAGE).convert("RGB")
rows, ref_pixels, ref_ids = [], None, None
for name, (fast, dev) in {"slow": (False, "cpu"), "fast-cpu": (True, "cpu"), "fast-gpu": (True, "cuda")}.items():
    proc = AutoProcessor.from_pretrained(MODEL_ID, use_fast=fast)
    kw = {} if dev == "cpu" else {"device": dev}
    tile_ms = mean_ms(lambda: proc.image_processor(images=[img], return_tensors="pt", **kw))
    total_ms = mean_ms(lambda: make_inputs(proc, IMAGE, PROMPT, dev))
    torch.cuda.reset_peak_memory_stats()
    inputs = make_inputs(proc, IMAGE, PROMPT, dev)
    peak = torch.cuda.max_memory_allocated() / 2**20
    pixels = inputs["pixel_values"].float().cpu()
    if ref_pixels is None:
        ref_pixels = pixels
    diff = (pixels - ref_pixels).abs()
    ids = dec.generate(inputs, eos)
    if ref_ids is None:
        ref_ids = ids
    rows.append({"option": name, "processor": type(proc.image_processor).__name__,
                 "tile+norm ms": round(tile_ms, 1), "input total ms": round(total_ms, 1),
                 "peak VRAM MB": round(peak), "pixel diff max": round(float(diff.max()), 3),
                 "pixel diff mean": round(float(diff.mean()), 4), "same answer": ids == ref_ids})
pd.DataFrame(rows)

## Log of previous runs

The cells below are the earlier notebook with their outputs. They call the scripts that were replaced by the code above, so they are not run again.

In [ ]:
REPO = "./edge-ai-smol-lvm-250-optimization"
!python {REPO}/gpu_preprocess.py --image {REPO}/frame.jpg

## Result

The test ran on a Colab GPU with one image. The input has 875 tokens (about 13 tiles).

| Option | Image processor | Tile + normalize (ms) | Total (ms) | Speed-up (total) | Peak VRAM (MB) | Pixel diff max / mean | Same answer |
|---|---|---|---|---|---|---|---|
| slow | Idefics3ImageProcessorPil | 286.9 | 299.1 | 1x | 622 | 0.000 / 0.0000 | True |
| fast-cpu | Idefics3ImageProcessor | 101.1 | 117.4 | 2.5x | 689 | 0.204 / 0.0052 | True |
| fast-gpu | Idefics3ImageProcessor | 10.3 | 11.6 | 26x | 768 | 0.204 / 0.0053 | True |

Total is the full input preparation: JPEG decode, tile, normalize, tokenize and copy to the GPU.

- `fast-gpu` takes 11.6 ms for the full input preparation. `slow` takes 299 ms. This saves about 287 ms for each request.
- The fast processor on the CPU is 2.5 times faster than `slow`. The GPU makes it 10 times faster again.
- The fast processors resize with BICUBIC and not LANCZOS (see the warning in the output). The pixel difference is small: 0.005 on average and 0.2 at most. Both fast options have the same difference, so the GPU adds no extra error.
- The model gives the same answer for all options. The test used one image only. Test more images before you use `fast-gpu`.
- `fast-gpu` uses 146 MB more peak VRAM than `slow` (768 MB and 622 MB).
- The earlier tests measured about 308 ms for tile and normalize. This was probably the `slow` (PIL) processor. With `fast-gpu`, preprocessing falls from about 29% to about 1.5% of the request (estimate).